In [ ]:
 key = "your-key-here"

In [ ]:
!pip install PyPDF2 nltk pandas numpy matplotlib seaborn scikit-learn wordcloud openpyxl


In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# For PDF processing
import PyPDF2

# For NLP
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer

# For TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# Downloading required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('omw-1.4')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
!pip install nltk

In [ ]:
# Simple function to grab synonyms from WordNet
def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonym = lemma.name().replace('_', ' ').lower()
            synonyms.add(synonym)
    return synonyms

# Add synonyms to our keyword list
def expand_keywords_with_synonyms(keywords_list, max_synonyms_per_word=3):
    expanded = set(keywords_list)

    for keyword in keywords_list:
        synonyms = get_synonyms(keyword)
        for syn in list(synonyms)[:max_synonyms_per_word]:
            if len(syn) > 2:  # skip super short words
                expanded.add(syn)

    return sorted(list(expanded))

# Starting keywords for each ESG category
BASE_ESG_KEYWORDS = {
    'Environmental': [
        'climate', 'carbon', 'emission', 'renewable', 'energy',
        'sustainability', 'environmental', 'waste', 'recycling', 'pollution',
        'biodiversity', 'ecosystem', 'water', 'greenhouse',
        'deforestation', 'conservation', 'footprint',
        'sustainable', 'ecological', 'nature', 'ocean', 'forest', 'plastic',
        'solar', 'wind', 'fossil', 'green'
    ],
    'Social': [
        'employee', 'diversity', 'inclusion', 'equity', 'workforce',
        'labor', 'community', 'safety', 'health', 'wellbeing',
        'training', 'development', 'gender', 'equality', 'discrimination',
        'workplace', 'welfare', 'social', 'stakeholder', 'engagement',
        'philanthropy', 'volunteering', 'hiring', 'retention', 'culture',
        'benefits', 'human', 'rights'
    ],
    'Governance': [
        'governance', 'board', 'director', 'ethics', 'compliance',
        'transparency', 'accountability', 'audit', 'risk', 'management',
        'shareholder', 'executive', 'compensation', 'integrity',
        'corruption', 'bribery', 'policy', 'regulation', 'regulatory',
        'oversight', 'independent', 'committee', 'disclosure', 'reporting',
        'ethical', 'leadership'
    ]
}

# Expand keywords with synonyms
print("Expanding keywords with synonyms:\n")
ESG_KEYWORDS = {}

for pillar, keywords in BASE_ESG_KEYWORDS.items():
    expanded = expand_keywords_with_synonyms(keywords, max_synonyms_per_word=2)
    ESG_KEYWORDS[pillar] = expanded
    print(f"{pillar}: {len(keywords)} → {len(expanded)} keywords")

# Add multi-word phrases (WordNet doesn't handle these well)
ADDITIONAL_PHRASES = {
    'Environmental': [
        'renewable energy', 'clean energy', 'net zero', 'decarbonization',
        'circular economy', 'resource efficiency', 'climate change',
        'carbon footprint', 'carbon neutral', 'carbon dioxide', 'co2',
        'ghg emissions', 'global warming', 'environmental impact'
    ],
    'Social': [
        'human rights', 'customer satisfaction', 'supply chain', 'fair trade',
        'living wage', 'local communities', 'work life balance', 'employee engagement',
        'diversity and inclusion', 'pay equity', 'occupational health', 'labor rights',
        'community engagement', 'social responsibility', 'fair labor'
    ],
    'Governance': [
        'corporate governance', 'board of directors', 'risk management',
        'executive compensation', 'conflict of interest', 'code of conduct',
        'internal controls', 'compliance program', 'board independence',
        'shareholder rights', 'corporate ethics', 'whistleblower protection',
        'anti corruption', 'data privacy', 'cyber security'
    ]
}

for pillar, phrases in ADDITIONAL_PHRASES.items():
    ESG_KEYWORDS[pillar].extend(phrases)
    ESG_KEYWORDS[pillar] = sorted(list(set(ESG_KEYWORDS[pillar])))

print(f"\nFinal counts:")
for pillar, keywords in ESG_KEYWORDS.items():
    print(f"  {pillar}: {len(keywords)} keywords")

# Preview some keywords
print(f"\nSample keywords per pillar:")
for pillar, keywords in ESG_KEYWORDS.items():
    print(f"\n{pillar}:")
    print(f"  {', '.join(keywords[:15])}...")

# Words that signal vague commitments / aspirational language (possible greenwashing)
GREENWASHING_INDICATORS = [
    'committed', 'commitment', 'dedication', 'dedicated', 'passionate', 'leading',
    'strive', 'striving', 'world-class', 'best-in-class', 'innovative', 'excellence',
    'endeavor', 'endeavoring', 'working towards', 'aiming', 'planning', 'intend',
    'aspire', 'aspiring', 'believe', 'proud', 'excited', 'promising', 'exploring',
    'journey', 'vision', 'ambition', 'passionate about', 'hope', 'hoping', 'desire',
    'seek', 'seeking', 'continue to', 'ongoing', 'long term', 'future',
    'sustainable', 'eco-friendly', 'green', 'natural', 'responsible', 'carbon neutral',
    'net-zero', 'climate positive', 'nature positive', 'low impact', 'low-carbon',
    'environmentally friendly', 'ethical', 'pioneering', 'championing', 'transforming',
    'driving change', 'positive contribution', 'making a difference', 'towards a better',
    'future-ready', 'sustainability journey', 'on track', 'progressing', 'advancing',
    'fostering', 'empowering', 'harnessing', 'leveraging', 'synergies', 'holistic'
]

# Optional: If you still want to use the expand function
GREENWASHING_INDICATORS = expand_keywords_with_synonyms(GREENWASHING_INDICATORS, max_synonyms_per_word=2)
print(f"\nGreenwashing indicators: {len(GREENWASHING_INDICATORS)} terms")

# Words that signal concrete action / metrics (substantive language)
SUBSTANTIVE_WORDS = [
    'achieved', 'reduced', 'increased', 'implemented', 'completed', 'delivered',
    'measured', 'reported', 'certified', 'audited', 'verified', 'reached',
    'target', 'goal', 'metric', 'data', 'performance', 'result', 'outcome',
    'baseline', 'benchmark', 'kpi', 'indicator', 'quantified', 'tracked',
    'invested', 'spent', 'allocated', 'million', 'billion', 'percent', 'percentage',
    'launched', 'established', 'created', 'installed', 'deployed', 'executed',
    'tonnes', 'kg', 'gj', 'kwh', 'mwh', 'scope 1', 'scope 2', 'scope 3',
    'ghg emissions', 'carbon emissions', 'energy consumption', 'water consumption',
    'waste generated', 'waste diverted', 'recycled', 'reused', 'landfill',
    'compared to previous year', 'year-over-year', 'reduction of', 'avoided',
    'capex', 'investment of', 'roadmap', 'action plan', 'third party', 'assured',
    'limited assurance', 'reasonable assurance', 'actual', 'figures', 'numbers',
    'disclosed', 'calculated', 'monitored', 'training hours', 'employee count'
]

SUBSTANTIVE_WORDS = expand_keywords_with_synonyms(SUBSTANTIVE_WORDS, max_synonyms_per_word=2)
print(f"Substantive action words: {len(SUBSTANTIVE_WORDS)} terms")

# Save to file for later reference
with open('expanded_esg_keywords.txt', 'w') as f:
    for pillar, keywords in ESG_KEYWORDS.items():
        f.write(f"\n{pillar.upper()} ({len(keywords)} keywords)\n")
        f.write("="*60 + "\n")
        f.write(', '.join(keywords) + '\n')

    f.write(f"\nGREENWASHING INDICATORS ({len(GREENWASHING_INDICATORS)} terms)\n")
    f.write("="*60 + "\n")
    f.write(', '.join(GREENWASHING_INDICATORS) + '\n')

    f.write(f"\nSUBSTANTIVE WORDS ({len(SUBSTANTIVE_WORDS)} terms)\n")
    f.write("="*60 + "\n")
    f.write(', '.join(SUBSTANTIVE_WORDS) + '\n')

print(f"\nKeywords saved to 'expanded_esg_keywords.txt'")

Expanding keywords with synonyms:

Environmental: 27 → 54 keywords
Social: 28 → 67 keywords
Governance: 26 → 68 keywords

Final counts:
  Environmental: 68 keywords
  Social: 82 keywords
  Governance: 83 keywords

Sample keywords per pillar:

Environmental:
  atomic number 6, biodiversity, body of water, carbon, carbon dioxide, carbon footprint, carbon neutral, circular economy, clean energy, climate, climate change, clime, co2, conservation, consume...

Social:
  benefits, betrothal, cellular inclusion, childbed, civilisation, civilization, community, community engagement, community of interests, culture, customer satisfaction, date, development, discrimination, diversity...

Governance:
  account, accountability, administrator, answerability, anti corruption, audit, autonomous, board, board independence, board of directors, bribery, citizens committee, code of conduct, commission, committee...

Greenwashing indicators: 133 terms
Substantive action words: 154 terms

Keywords saved to 

In [ ]:
# Extract text from PDF file
def extract_text_from_pdf(pdf_path):

    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            num_pages = len(pdf_reader.pages)

            for page_num in range(num_pages):
                page = pdf_reader.pages[page_num]
                page_text = page.extract_text()
                if page_text:
                    text += page_text + " "

        return text, num_pages

    except Exception as e:
        print(f"Error reading {pdf_path}: {str(e)}")
        return "", 0


def clean_text(text):
    if not text:
        return ""

    # 1. Normalize whitespace (removes tabs, newlines, and multiple spaces)
    text = re.sub(r'\s+', ' ', text)

    # 2. Remove non-printable/ASCII characters (fixes weird PDF symbols)
    text = text.encode("ascii", "ignore").decode()

    # 3. Keep essential punctuation and alphanumeric characters
    text = re.sub(r'[^a-zA-Z0-9\s\.\,\%\-\$]', ' ', text)

    # 4. Optional: Remove very short "junk" words (like page numbers or fragments)
    # text = re.sub(r'\b\w{1}\b', '', text)

    return text.strip()

In [ ]:
# Detect greenwashing by comparing aspirational vs concrete language
def detect_greenwashing(text):

    text_lower = text.lower()
    words = word_tokenize(text_lower)
    total_words = len(words)

    if total_words == 0:
        return {
            'greenwashing_indicators': 0,
            'substantive_words': 0,
            'greenwashing_density': 0,
            'substantive_density': 0,
            'greenwashing_ratio': 0,
            'risk_level': 'Unknown',
            'risk_score': 0
        }

    # Count vague/aspirational language
    greenwashing_count = 0
    for indicator in GREENWASHING_INDICATORS:
        greenwashing_count += text_lower.count(indicator)

    # Count concrete actions/metrics
    substantive_count = 0
    for word in SUBSTANTIVE_WORDS:
        substantive_count += text_lower.count(word)

    # Calculate density per 1000 words
    greenwashing_density = (greenwashing_count / total_words) * 1000
    substantive_density = (substantive_count / total_words) * 1000

    # Calculate ratio: high ratio = more fluff than substance
    if substantive_count > 0:
        greenwashing_ratio = greenwashing_count / substantive_count
    else:
        greenwashing_ratio = greenwashing_count if greenwashing_count > 0 else 0

    # Risk assessment
    risk_score = (greenwashing_ratio * 0.64) + (greenwashing_density * 0.36)


    if greenwashing_ratio > 1.5 or greenwashing_density > 18:
        risk_level = 'High'
    elif greenwashing_ratio > 1 or greenwashing_density > 10:
        risk_level = 'Medium'
    else:
        risk_level = 'Low'

    return {
        'greenwashing_indicators': greenwashing_count,
        'substantive_words': substantive_count,
        'greenwashing_density': round(greenwashing_density, 2),
        'substantive_density': round(substantive_density, 2),
        'greenwashing_ratio': round(greenwashing_ratio, 2),
        'risk_level': risk_level,
        'risk_score': round(risk_score, 2)
    }

In [ ]:
# Calculate which ESG pillars the company focuses on
def calculate_esg_importance(text, company_name="Company"):

    text_lower = text.lower()

    # Count keyword mentions for each pillar
    esg_counts = {}
    for pillar, keywords in ESG_KEYWORDS.items():
        count = 0
        for keyword in keywords:
            count += text_lower.count(keyword.lower())
        esg_counts[pillar] = count

    total_keywords = sum(esg_counts.values())

    if total_keywords == 0:
        return {
            'Environmental_count': 0,
            'Social_count': 0,
            'Governance_count': 0,
            'Environmental_pct': 0,
            'Social_pct': 0,
            'Governance_pct': 0,
            'dominant_pillar': 'None',
            'total_esg_keywords': 0
        }

    # Calculate percentages
    env_pct = (esg_counts['Environmental'] / total_keywords) * 100
    soc_pct = (esg_counts['Social'] / total_keywords) * 100
    gov_pct = (esg_counts['Governance'] / total_keywords) * 100

    # Find dominant pillar
    dominant_pillar = max(esg_counts.items(), key=lambda x: x[1])[0]

    return {
        'Environmental_count': esg_counts['Environmental'],
        'Social_count': esg_counts['Social'],
        'Governance_count': esg_counts['Governance'],
        'Environmental_pct': round(env_pct, 2),
        'Social_pct': round(soc_pct, 2),
        'Governance_pct': round(gov_pct, 2),
        'dominant_pillar': dominant_pillar,
        'total_esg_keywords': total_keywords
    }

In [ ]:
# Using TF-IDF to compare ESG focus across companies
def tfidf_esg_analysis(texts_dict):

    companies = list(texts_dict.keys())
    documents = list(texts_dict.values())

    results = []

    for pillar, keywords in ESG_KEYWORDS.items():
        # TF-IDF with ESG keywords as vocabulary
        vectorizer = TfidfVectorizer(
            vocabulary=keywords,
            lowercase=True,
            token_pattern=r'\b\w+\b'
        )

        try:
            tfidf_matrix = vectorizer.fit_transform(documents)

            # Get scores for each company
            for idx, company in enumerate(companies):
                doc_scores = tfidf_matrix[idx].toarray().flatten()
                mean_tfidf = np.mean(doc_scores) if len(doc_scores) > 0 else 0
                max_tfidf = np.max(doc_scores) if len(doc_scores) > 0 else 0

                results.append({
                    'company': company,
                    'pillar': pillar,
                    'tfidf_mean': round(mean_tfidf, 4),
                    'tfidf_max': round(max_tfidf, 4)
                })

        except Exception as e:
            print(f"TF-IDF issue for {pillar}: {e}")
            for company in companies:
                results.append({
                    'company': company,
                    'pillar': pillar,
                    'tfidf_mean': 0,
                    'tfidf_max': 0
                })

    return pd.DataFrame(results)

In [ ]:
def return_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def analyze_single_esg_report(pdf_path, company_name, industry, controversy_level="Low"):


    print(f"\n{'='*60}")
    print(f"Analyzing: {company_name}")
    print(f"Industry: {industry} | Controversy: {controversy_level}")
    print(f"{'='*60}")

    """
    text, num_pages = extract_text_from_pdf(pdf_path)
    if not text or len(text) < 100:
        print("Failed to extract text")
        return
    """

    #clean_text_content = clean_text(text)
    clean_text_content = return_text(pdf_path)
    global text
    text = clean_text_content

    '''
    with open('cleaned_report.txt', 'w') as f:
      f.write(clean_text_content)
    '''

    word_count = len(word_tokenize(clean_text_content))

    print(f"Word count: {word_count:,}")

    # Greenwashing detection
    greenwashing = detect_greenwashing(clean_text_content)
    print(f"\nGreenwashing:")
    print(f"  Risk: {greenwashing['risk_level']}")
    print(f"  Ratio: {greenwashing['greenwashing_ratio']:.2f}")
    print(f"  Aspirational words: {greenwashing['greenwashing_indicators']}")
    print(f"  Substantive words: {greenwashing['substantive_words']}")

    # ESG pillar analysis
    esg_importance = calculate_esg_importance(clean_text_content, company_name)
    print(f"\nESG Focus:")
    print(f"  Environmental: {esg_importance['Environmental_pct']:.1f}%")
    print(f"  Social: {esg_importance['Social_pct']:.1f}%")
    print(f"  Governance: {esg_importance['Governance_pct']:.1f}%")
    print(f"  Dominant: {esg_importance['dominant_pillar']}")

    # Compile results
    results = {
        'company_name': company_name,
        'industry': industry,
        'controversy_level': controversy_level,
        'word_count': word_count,
        'text_content': clean_text_content,
    }

    results.update(greenwashing)
    results.update(esg_importance)

    print(f"\nAnalysis complete for {company_name}\n")

    return results

In [ ]:
def analyze_industry_reports(pdf_folder, industry_name, company_info):

    all_results = []

    for filename, (company_name, controversy) in company_info.items():
        pdf_path = os.path.join(pdf_folder, filename)

        if not os.path.exists(pdf_path):
            print(f"File not found: {pdf_path}")
            continue

        result = analyze_single_esg_report(pdf_path, company_name, industry_name, controversy)

        if result:
            all_results.append(result)

    df = pd.DataFrame(all_results)

    print(f"\n{'='*60}")
    print(f"Analysis complete: {industry_name}")
    print(f"Reports analyzed: {len(all_results)}")
    print(f"{'='*60}\n")

    return df


In [ ]:
def analyze_industry_reports(pdf_folder, industry_name, company_info):

    all_results = []

    for filename, (company_name, controversy) in company_info.items():
        pdf_path = os.path.join(pdf_folder, filename)

        if not os.path.exists(pdf_path):
            print(f"File not found: {pdf_path}")
            continue

        result = analyze_single_esg_report(pdf_path, company_name, industry_name, controversy)

        if result:
            all_results.append(result)

    df = pd.DataFrame(all_results)

    return df


In [ ]:
PDF_FOLDER = ""
INDUSTRY_NAME = "Racing/Sports"
txt_path = "NASCAR.txt"
company_name = "NASCAR"
controversy_level = "Unknown"

COMPANY_INFO = {
    txt_path: (company_name, controversy_level)
}


print("Configuration:")
print(f"  Industry: {INDUSTRY_NAME}")
print(f"  PDF folder: {PDF_FOLDER}")
print(f"  Companies: {len(COMPANY_INFO)}")
print(f"\nCompanies to analyze:")
for filename, (company, controversy) in COMPANY_INFO.items():
    print(f"  - {company} ({controversy} controversy)")

Configuration:
  Industry: Racing/Sports
  PDF folder: 
  Companies: 1

Companies to analyze:
  - NASCAR (Unknown controversy)


In [ ]:
# Run complete analysis
import nltk
nltk.download('punkt_tab')
results_df = analyze_industry_reports(PDF_FOLDER, INDUSTRY_NAME, COMPANY_INFO)

# Show results preview
print("\n" + "="*60)
print("RESULTS PREVIEW")
print("="*60)
display(results_df.head())

GW_ratio, GW_density = float(results_df['greenwashing_ratio'][0]), float(results_df['greenwashing_density'][0])

# Show key columns
print("\nKey Metrics:")
key_cols = ['company_name', 'controversy_level',
            'risk_level', 'greenwashing_ratio', 'dominant_pillar']
display(results_df[key_cols])


Analyzing: NASCAR
Industry: Racing/Sports | Controversy: Unknown
Word count: 1,720

Greenwashing:
  Risk: High
  Ratio: 0.90
  Aspirational words: 38
  Substantive words: 42

ESG Focus:
  Environmental: 30.9%
  Social: 56.7%
  Governance: 12.4%
  Dominant: Social

Analysis complete for NASCAR


RESULTS PREVIEW


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,company_name,industry,controversy_level,word_count,text_content,greenwashing_indicators,substantive_words,greenwashing_density,substantive_density,greenwashing_ratio,risk_level,risk_score,Environmental_count,Social_count,Governance_count,Environmental_pct,Social_pct,Governance_pct,dominant_pillar,total_esg_keywords
0,NASCAR,Racing/Sports,Unknown,1720,NASCAR IMPACT 2025 REPORT\n===================...,38,42,22.11,24.43,0.9,High,8.54,30,55,12,30.93,56.7,12.37,Social,97



Key Metrics:


,company_name,controversy_level,risk_level,greenwashing_ratio,dominant_pillar
0,NASCAR,Unknown,High,0.9,Social


In [ ]:
from transformers import pipeline
import torch
import re
import numpy as np
from pathlib import Path
device = 0 if torch.cuda.is_available() else -1
sentiment_pipe = pipeline(
    "text-classification",
    model="climatebert/distilroberta-base-climate-sentiment",
    device=device
)
commitment_pipe = pipeline(
    "text-classification",
    model="climatebert/distilroberta-base-climate-commitment",
    device=device
)
specificity_pipe = pipeline(
    "text-classification",
    model="climatebert/distilroberta-base-climate-specificity",
    device=device
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: climatebert/distilroberta-base-climate-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: climatebert/distilroberta-base-climate-commitment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: climatebert/distilroberta-base-climate-specificity
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def load_keywords_from_file(filename="expanded_esg_keywords.txt"):
    keywords = {
        'GREENWASHING_INDICATORS': [],
        'SUBSTANTIVE_WORDS': [],
        'HEDGING_PHRASES': []
    }

    try:
        with open(filename, 'r', encoding='utf-8') as f:
            content = f.read()

        # Extract GREENWASHING INDICATORS
        if "GREENWASHING INDICATORS" in content:
            section = content.split("GREENWASHING INDICATORS")[1].split("SUBSTANTIVE WORDS")[0]
            lines = section.strip().split('\n')
            for line in lines:
                if line.strip() and not line.startswith("="):
                    keywords['GREENWASHING_INDICATORS'].extend([w.strip() for w in line.split(',') if w.strip()])

        # Extract SUBSTANTIVE WORDS
        if "SUBSTANTIVE WORDS" in content:
            section = content.split("SUBSTANTIVE WORDS")[1]
            lines = section.strip().split('\n')
            for line in lines:
                if line.strip() and not line.startswith("="):
                    keywords['SUBSTANTIVE_WORDS'].extend([w.strip() for w in line.split(',') if w.strip()])

        print(f"Loaded {len(keywords['GREENWASHING_INDICATORS'])} greenwashing indicators")
        print(f"Loaded {len(keywords['SUBSTANTIVE_WORDS'])} substantive words")

    except Exception as e:
        print(f"Warning: Could not load keywords from file: {e}")
        # Fallback hedging list if file reading fails
        keywords['HEDGING_PHRASES'] = [
            "aims to", "strives for", "working towards", "seeks to", "intends to",
            "hopes to", "approximately", "potentially", "may", "might", "could",
            "on track to", "journey towards", "progress towards", "ambition to"
        ]

    return keywords

def count_hedging(text: str, hedging_list) -> float:
    if not text or not hedging_list:
        return 0.0
    text_lower = text.lower()
    count = sum(text_lower.count(phrase) for phrase in hedging_list)
    sentences = max(1, len(re.split(r'[.!?]+', text)))
    return min(1.0, count / (sentences * 0.6))

def analyze_report(clean_text):
    print(f"Loaded cleaned report: {len(clean_text):,} characters\n")

    # Load keywords
    kw = load_keywords_from_file()

    hedging_list = [
        "aims to", "strives for", "working towards", "seeks to", "intends to",
        "hopes to", "approximately", "potentially", "on track to", "journey towards"
    ]

    # Split into sentences
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?|\!)\s', clean_text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 20]

    print(f"Found {len(sentences)} sentences. Processing up to 600...\n")

    sent_scores, comm_scores, spec_scores, hedging_flags = [], [], [], []
    processed = 0

    for sent in sentences[:600]:
        # Truncate sentence if too long for the model (512 tokens)
        tokens = sent.split()
        if len(tokens) > 450:   # safe margin
            sent = " ".join(tokens[:450])

        try:
            sent_res = sentiment_pipe(sent)[0]
            comm_res = commitment_pipe(sent)[0]
            spec_res = specificity_pipe(sent)[0]

            # Extract floats correctly
            sent_pos = sent_res['score'] if sent_res['label'].lower() in ['opportunity', 'positive'] else 0.0
            comm_prob = comm_res['score'] if comm_res['label'] == 'yes' else (1.0 - comm_res['score'])

            # If label is 'spec', use score. If 'non', use (1-score) to get specificity probability
            spec_prob = spec_res['score'] if spec_res['label'] == 'spec' else (1.0 - spec_res['score'])

            hedge = count_hedging(sent, hedging_list)

            # Append the FLOAT variables, not the DICTS
            sent_scores.append(sent_pos)
            comm_scores.append(comm_prob)
            spec_scores.append(spec_prob)
            hedging_flags.append(hedge)

            processed += 1

            if processed % 100 == 0:
                print(f"Processed {processed} sentences...")

        except Exception as e:
            continue

    if not sent_scores:
        print("No sentences processed successfully.")
        return

  # Aggregate
    avg_sent = np.mean(sent_scores)
    avg_comm = np.mean(comm_scores)
    avg_spec = np.mean(spec_scores)
    avg_hedge = np.mean(hedging_flags)

    gw_bert_risk = 0.71 * avg_sent + 0.14 * avg_comm - 0.86 * avg_spec - 0.71 * avg_hedge
    gw_bert_risk = - gw_bert_risk if gw_bert_risk < 0 else gw_bert_risk


    if gw_bert_risk >= 0.5:
        risk_category = "High"
    elif gw_bert_risk >= 0.30:
        risk_category = "Medium"
    else:
        risk_category = "Low"

    global BERT_score
    BERT_score = gw_bert_risk

    # Final Output
    print("="*60)
    print("CLIMATEBERT + HEDGING ANALYSIS RESULTS")
    print("="*60)
    print(f"Risk Category                : {risk_category}")
    print(f"BERT Greenwashing Risk Score : {gw_bert_risk:.3f} / 1.0")
    print(f"Avg Positive Sentiment       : {avg_sent:.3f}")
    print(f"Avg Commitment               : {avg_comm:.3f}")
    print(f"Avg Specificity              : {avg_spec:.3f}")
    print(f"Avg Hedging Ratio            : {avg_hedge:.3f}")
    print(f"Sentences Analyzed           : {len(sent_scores)}")
    print("="*60)

analyze_report(text)

Loaded cleaned report: 9,473 characters

Loaded 134 greenwashing indicators
Loaded 155 substantive words
Found 50 sentences. Processing up to 600...



Token indices sequence length is longer than the specified maximum sequence length for this model (989 > 512). Running this sequence through the model will result in indexing errors


CLIMATEBERT + HEDGING ANALYSIS RESULTS
Risk Category                : Low
BERT Greenwashing Risk Score : 0.090 / 1.0
Avg Positive Sentiment       : 0.520
Avg Commitment               : 0.433
Avg Specificity              : 0.395
Avg Hedging Ratio            : 0.000
Sentences Analyzed           : 49


SHORT NOTES:

  "overall_verdict": for ablation

  "scrutiny_confidence": for model scoring

  "deflection_analysis": ablation

  "language_analysis": ablation

  "key_red_flags": check with reasons

  "final_take": check with reasons.



  get screenshots for lexicon, bert and llm model responses, and final score

In [ ]:
report_text = text

from openai import OpenAI
import json

# Initialize OpenRouter client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=key,
)

system_prompt = """You are a harsh, no-nonsense business and sustainability critic.
Your job is to deeply scrutinize companies and call out any shady, misleading, or overly polished practices in their BRSR or ESG reports.
You do not trust corporate language easily. You look for substance and use reasoning along with your internal knowledge about the company's past if available.

Company: {company_name}
Industry/Domain: {industry}

Task:
Read the full report carefully and give a brutally honest assessment.

Focus especially on:
1. Deflection / Social Washing: In industries like FMCG, Consumer Goods, or Personal Care, companies often over-emphasize Social topics to deflect from environmental impacts of their core products (water use, plastics, chemicals, Scope 3).
2. Aspirational vs Substantive Language: Look for too many vague buzzwords, journeys, ambitions, and future promises vs actual concrete metrics and actions.
3. Vague Future-Talk and Selective Disclosure.
4. Any inconsistencies or buried issues.

Return your analysis **strictly as valid JSON** with no extra text:

{{
  "overall_verdict": "High Risk" | "Medium Risk" | "Low Risk" | "Clean",
  "scrutiny_confidence": "0-1",
  "deflection_analysis": "Short explanation focusing on industry context and possible social washing",
  "language_analysis": "Short explanation about aspirational vs substantive language",
  "key_red_flags": ["flag 1", "flag 2", "flag 3"] (if any, else return []),
  "final_take": "2-3 direct and harsh sentences"
}}
"""



messages = [
    {
        "role": "system",
        "content": system_prompt.format(
            company_name=company_name,
            industry=INDUSTRY_NAME
        )
    },
    {
        "role": "user",
        "content": f"Here is the full BRSR/ESG report text to analyze:\n\n{report_text}"
    }
]


completion = client.chat.completions.create(
    model="google/gemma-4-26b-a4b-it:free",
    messages=messages,
    temperature=0.3,
    max_tokens=2000,
    response_format={"type": "json_object"}
)


try:
    result = json.loads(completion.choices[0].message.content)
    print(json.dumps(result, indent=2))
except json.JSONDecodeError:
    print("Failed to parse JSON. Raw output:")
    print(completion.choices[0].message.content)


{
  "overall_verdict": "High Risk",
  "scrutiny_confidence": "0.95",
  "deflection_analysis": "Classic social washing. The report leans heavily into emotional storytelling regarding the France family legacy, youth health, and military support to distract from the massive environmental footprint inherent in a high-octane, fossil-fuel-dependent motorsport. They are using $50 million in historical philanthropy to mask a complete lack of current, granular environmental data.",
  "language_analysis": "Extremely high density of aspirational fluff. Terms like 'harness the thrill,' 'driving a better world,' and 'sparking interest' are used to fill the void where actual performance metrics should be. The report is a collection of 'journeys' and 'ambitions' rather than a record of achievements.",
  "key_red_flags": [
    "Absence of Scope 3 emissions data, which is the most critical metric for a racing industry reliant on fuel and logistics.",
    "Vague environmental targets: 'Comprehensive was

In [ ]:
lexicon_score = 0.65 * GW_ratio + 0.35 * GW_density
lexicon_score /= 10
lexicon_score = max(0.0, min(1.0, lexicon_score))

In [ ]:
LLM_score = result['scrutiny_confidence']
LLM_score = float(LLM_score)
LLM_score

0.95

In [ ]:
w1 = 0.25
w2 = 0.23
w3 = 0.52

finalized_score = w1 * lexicon_score + w2 * BERT_score + w3 * LLM_score
finalized_score

np.float64(0.7227879704868306)

In [ ]:
if finalized_score >= 0.66:
    risk_category_final = "High"
elif finalized_score >= 0.33:
    risk_category_final = "Medium"
else:
    risk_category_final = "Low"
risk_category_final

'High'

In [ ]:
red_flags = result['key_red_flags']
red_flags

['Absence of Scope 3 emissions data, which is the most critical metric for a racing industry reliant on fuel and logistics.',
 "Vague environmental targets: 'Comprehensive waste reduction plans' and 'publishing a plan' are not actions; they are promises to eventually make promises.",
 'Extreme temporal distance: Most major environmental commitments (Net Zero, Carbon Reduction Plan) are set for 2030 or 2035, providing a decade-long shield against accountability.',
 'Selective disclosure: The report provides historical cumulative data for the Foundation (which is easy to track) but zero real-time data for current environmental impact or carbon intensity.']

In [ ]:
reasoning = result['final_take']
reasoning

"This is a marketing brochure masquerading as a sustainability report. NASCAR is attempting to buy social license through philanthropic nostalgia while kicking the environmental can down a decade-long road. If you can't even measure your carbon footprint until 2030, you aren't 'leading the way'—you're just stalling."